# M10 — protótipo de treino (Colab Pro)

**O objetivo deste notebook não é um modelo bom. É um número:** o **custo por época**.

O plano de treino (`docs/plans/m10-treino-vastai.md`) estima a Fase 3 em $600–1.500 por
proporção ao run de M5 — cujo custo real **não está registrado em lugar nenhum do repositório**.
Esse é o risco R1, e é o número mais frágil de todo o plano. Um run de ~300 h aqui mede o custo
por época de verdade e permite reprecificar antes de comprometer centenas de dólares.

## O que este notebook NÃO faz

- não treina até convergir (são poucas épocas, de propósito)
- não decide a arquitetura — isso é o bake-off de T4
- não produz artefato publicável

## Configuração alvo

| | |
|---|---|
| GPU | **L4** (melhor custo/hora normalizado no Colab Pro) |
| Corpus | ~300 h do TAGARELA (~31 shards) |
| Features | fbank 80-dim em **lilcom_chunky** — 33 MB/h `[MEDIDO]` |
| Disco | ~10 GB de features + ~17 GB de parquets temporários |
| Checkpoints | no Drive, para sobreviver à sessão |


## 1. Ambiente

Colab Pro **não tem background execution** (isso é Pro+). A aba precisa ficar aberta.
Com checkpoint por época, uma desconexão vira atraso, não perda.


In [ ]:
import subprocess, sys, os, json, time
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip())
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| disponivel', torch.cuda.is_available())
print('disco livre:', subprocess.run(['df','-h','/content'], capture_output=True, text=True).stdout.splitlines()[-1])


## 2. Dependências

⚠️ **O ponto mais frágil do notebook é o `k2`**: a wheel precisa casar exatamente com a versão
de torch e CUDA do runtime. Se a célula falhar, confira a matriz em
<https://k2-fsa.github.io/k2/installation/pre-compiled-cuda-wheels-linux/> e ajuste a URL.

O `k2stub` do repositório **não serve aqui** — ele cobre só a ativação Swoosh para smoke em
CPU, e a receita de treino usa `k2.ctc_loss` e grafos.


In [ ]:
# k2 precisa casar torch+cuda do runtime; a URL abaixo e o indice oficial de wheels
TORCH = torch.__version__.split('+')[0]
CUDA  = (torch.version.cuda or '').replace('.', '')
print(f'procurando k2 para torch {TORCH} cuda {CUDA}')
!pip install -q k2 -f https://k2-fsa.github.io/k2/cuda.html || echo 'AJUSTAR A WHEEL MANUALMENTE'
!pip install -q lhotse kaldialign sentencepiece kaldi_native_fbank num2words regex
import k2; print('k2', k2.__version__)


## 3. Código: icefall + jvscribe

O `jvscribe` traz os preparadores de corpus já testados (`prep_tagarela.py`), com filtro
determinístico de alucinação e relatório de cobertura por show.


In [ ]:
%cd /content
!test -d icefall  || git clone -q https://github.com/k2-fsa/icefall.git
!test -d jvscribe || git clone -q https://github.com/usetheodev/jvscribe.git
os.environ['PYTHONPATH'] = '/content/icefall:' + os.environ.get('PYTHONPATH','')
!pip install -q -r /content/icefall/requirements.txt 2>/dev/null | tail -2
print('ok')


## 4. Checkpoints no Drive

A sessão morre; o Drive não. **Só os checkpoints vão para o Drive** — as features ficam no
disco local, que é ordens de magnitude mais rápido para o dataloader.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
EXP = '/content/drive/MyDrive/jvscribe/m10_proto/exp'
os.makedirs(EXP, exist_ok=True)
print('checkpoints ->', EXP)


## 5. Corpus

Amostragem **estratificada**: shards espaçados uniformemente ao longo dos 1.764. Como
**um shard é um show** (`wiki/medicoes/m10-t3-quanto-corpus-pt-existe.md`), espaçar por
posição é espaçar por show.

Cada shard tem ~9,62 h `[MEDIDO]`, então 31 shards ≈ 300 h.


In [ ]:
N_SHARDS = 31           # ~300 h; cada shard ~9,62 h [MEDIDO]
TOTAL_SHARDS = 1764
RAW = '/content/tagarela_raw'; os.makedirs(RAW, exist_ok=True)
from huggingface_hub import hf_hub_download
idx = [round(i*(TOTAL_SHARDS-1)/(N_SHARDS-1)) for i in range(N_SHARDS)]
t0 = time.time()
for k, i in enumerate(idx):
    f = f'data/train-{i:05d}-of-01764.parquet'
    hf_hub_download('freds0/TAGARELA', f, repo_type='dataset', local_dir=RAW)
    if (k+1) % 5 == 0: print(f'  {k+1}/{N_SHARDS}  {time.time()-t0:.0f}s', flush=True)
print(f'{N_SHARDS} shards em {(time.time()-t0)/60:.1f} min')


### 5.1 Cuts + fbank

`prep_tagarela.py` filtra alucinação, gera o relatório de cobertura por show e grava features
em **lilcom_chunky** — 33 MB/h contra 115 MB/h do default numpy `[MEDIDO]`.


In [ ]:
!cd /content/jvscribe && python3 jvscribe/finetune/prep_tagarela.py \
    --parquet-dir /content/tagarela_raw/data \
    --out /content/data/tagarela \
    --num-jobs 4
!du -sh /content/data/tagarela/feats_train
!cat /content/data/tagarela/show_distribution.txt 2>/dev/null | head -5


### 5.2 Verificar o storage — não confie no parâmetro

O lhotse aceitou `storage_type` e gravou numpy assim mesmo numa execução real
(`wiki/medicoes/m10-t3-fbank-e-storage.md`). **Conferir custa um segundo; errar custa 410 GB**
na escala de 5.000 h.


In [ ]:
import gzip, json as _json
with gzip.open('/content/data/tagarela/tagarela_cuts_train.jsonl.gz','rt') as f:
    c = _json.loads(f.readline())
st = c['features']['storage_type']
print('storage_type =', st)
assert st == 'lilcom_chunky', f'ESPERADO lilcom_chunky, veio {st} -- 3,5x mais disco'


## 6. Tokenizer BPE

⚠️ O `bpe.model` do M4 **se perdeu** e custou retrabalho. Aqui ele vai para o Drive junto dos
checkpoints, porque sem ele o checkpoint é inútil.


In [ ]:
LANG = '/content/data/lang_bpe_500'; os.makedirs(LANG, exist_ok=True)
import sentencepiece as spm
spm.SentencePieceTrainer.train(
    input='/content/data/tagarela/transcript_words.txt',
    model_prefix=f'{LANG}/bpe', vocab_size=500, model_type='bpe',
    character_coverage=1.0, bos_id=-1, eos_id=-1, unk_id=2, pad_id=0)
!cp {LANG}/bpe.model {EXP}/   # sobrevive a sessao
print('bpe.model ->', EXP)


## 7. Treino instrumentado

As armadilhas já pagas entram como configuração, não como descoberta
(`wiki/treino/armadilhas.md`):

| armadilha | configuração |
|---|---|
| LR de cabeça fresca `0.0001` → WER 100% | `--base-lr 0.03` na cabeça |
| fp16 colapsa sob augmentação | `--use-fp16 0` |
| as 4 flags não estão no `.pt` | passadas explicitamente |
| disco cheio trunca o checkpoint **sem erro** | poda + verificação de tamanho |

**A instrumentação é o produto deste notebook**: segundos por época, e o custo em unidades.


In [ ]:
UNIDADES_POR_HORA = {'L4': 4.8, 'A100': 13.0, 'T4': 2.0}   # [ESTIMATIVA] nao medido
GPU = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
un_h = next((v for k,v in UNIDADES_POR_HORA.items() if k in GPU), None)
print(f'GPU: {GPU} | unidades/h: {un_h}')

EPOCAS = 3        # suficiente para medir custo/epoca; NAO para convergir
MAX_DUR = 300     # segundos por batch; baixar se der OOM

cmd = f'''cd /content/icefall/egs/commonvoice/ASR && python3 zipformer/train.py \
  --world-size 1 --num-epochs {EPOCAS} --start-epoch 1 \
  --exp-dir {EXP} --bpe-model {LANG}/bpe.model \
  --max-duration {MAX_DUR} --use-fp16 0 \
  --num-encoder-layers 2,2,3,4,3,2 \
  --feedforward-dim 512,768,1024,1536,1024,768 \
  --encoder-dim 192,256,384,512,384,256 \
  --encoder-unmasked-dim 192,192,256,256,256,192 \
  --causal 1 --use-ctc 1 --use-transducer 0 --base-lr 0.03'''
print(cmd)


In [ ]:
t0 = time.time()
!{cmd}
el = time.time() - t0
h = el/3600
print(f'\n=== CUSTO MEDIDO ===')
print(f'{EPOCAS} epocas em {h:.2f} h -> {h/EPOCAS:.3f} h por epoca')
if un_h:
    print(f'custo: {h*un_h:.1f} unidades = ${h*un_h/100*10:.2f}  ->  ${h/EPOCAS*un_h/100*10:.2f} por epoca')
    for horas_corpus in (1500, 5000):
        fator = horas_corpus/300
        print(f'  extrapolado p/ {horas_corpus} h x 10 epocas: '
              f'${h/EPOCAS*un_h/100*10*fator*10:.0f}  [ESTIMATIVA linear]')


## 8. O que fazer com o número

1. Registrar em `wiki/medicoes/` com a GPU, o corpus e as épocas ao lado — sem isso o número
   não transfere.
2. Reprecificar a Fase 3 de `docs/plans/m10-treino-vastai.md`, substituindo a proporção
   herdada de M5 pela medição.
3. **A extrapolação linear é otimista**: batch maior aproveita melhor a GPU, e corpus maior
   muda o gargalo de compute para I/O. Tratar como piso, não como estimativa.

### Se der OOM

Baixar `--max-duration` antes de qualquer outra coisa. O registro de M5 tem OOM em pre-scan
resolvido com `eager` + `workers=2`.

### Se a loss platôar em blank e o WER for 100%

É a armadilha do LR de cabeça fresca. Conferir que `--base-lr` está em 0.03 e não em 0.0001.
